In [3]:
#Basic FillNa 0
fillna_0=["Option_CT",
"Option_FL",
"Option_ICTQ",
"Option_WBQ",
"Option_PQ",
"Option_TQ",
"Option_UH",
"EFFORT1",
"EFFORT2",
"IMMIG",
"MISSSC"
]
df[fillna_0]=df[fillna_0].fillna(0)

final_cols += ["Option_CT",
"Option_FL",
"Option_ICTQ",
"Option_WBQ",
"Option_PQ",
"Option_TQ",
"Option_UH",
"MISSSC"
]
to_rs += ["EFFORT1",
"EFFORT2",
"IMMIG",
]

In [6]:
isco_conv = pd.read_csv("../../data/isco08_isei08_full_with_nearest.csv", sep=",",index_col=0)
isco_dict = isco_conv.to_dict()["isei08"]
ocods=["OCOD1","OCOD2","OCOD3"]
for c in ocods:
    df[c]=df[c].map(isco_dict)


In [8]:
df["AGE"]=df["AGE"].fillna(df["Year"]-df["ST003D03T"])
df["AGE"]=df["AGE"] - df.groupby(["Year","CNTSCHID"])["AGE"].transform("mean")

to_rs.append("AGE")

In [10]:
COBN_S_fil_1 = df.groupby(["CNTSCHID", "Year"])["COBN_S"] \
    .transform(lambda s: s.mode().iloc[0] if not s.mode().empty else s.mean())

COBN_S_fil_2 = df.groupby("CNTSCHID")["COBN_S"] \
    .transform(lambda s: s.mode().iloc[0] if not s.mode().empty else s.mean())

COBN_S_fil_3 = df["COBN_S"].mean()

# Chaîne d’imputation
df["COBN_S"] = (
    df["COBN_S"]
      .fillna(COBN_S_fil_1)
      .fillna(COBN_S_fil_2)
      .fillna(COBN_S_fil_3)
)

to_te.append("COBN_S")


In [12]:
#Fin Fill na, création de feature

In [14]:
df["DIFFERENT"] =(df["LANGTEST_COG"] != df.groupby(["CNTSCHID"])["LANGTEST_COG"]
.transform(lambda s: s.mode().iloc[0])).astype(int)

final_cols.append("DIFFERENT")

In [16]:
df["ADMINMODE"] = (df["ADMINMODE"]-1)
final_cols.append("ADMINMODE")

In [19]:
from collections import defaultdict

# ------------------------------------------
# 1. distance de Levenshtein
# ------------------------------------------
def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)

    if len(a) > len(b):
        a, b = b, a

    previous_row = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current_row = [i]
        for j, cb in enumerate(b, start=1):
            insert_cost  = current_row[j - 1] + 1
            delete_cost  = previous_row[j] + 1
            replace_cost = previous_row[j - 1] + (ca != cb)
            current_row.append(min(insert_cost, delete_cost, replace_cost))
        previous_row = current_row
    return previous_row[-1]

# ------------------------------------------
# 2. pré-calculs : clés du dict + fréquences
# ------------------------------------------
keys = list(stratum_conv_dict.keys())

def get_prefix(code: str, n: int = 3) -> str:
    s = str(code)
    return s[:n] if len(s) >= n else s

from collections import defaultdict
keys_by_prefix = defaultdict(list)
for k in keys:
    prefix = get_prefix(k)
    keys_by_prefix[prefix].append(k)

stratum_freq = df["STRATUM"].value_counts().to_dict()

# ------------------------------------------
# 3. fonction de résolution (NaN -> 5)
# ------------------------------------------
def closest_key(code, max_dist=4):
    """
    Retourne la valeur de stratum_conv_dict associée à la clé la plus proche.
    Si aucune clé trouvée à distance ≤ max_dist -> retourne 5.
    """
    if pd.isna(code):
        return 5   # au lieu de np.nan

    s = str(code)

    # correspondance exacte
    if s in stratum_conv_dict:
        return stratum_conv_dict[s]

    # candidats : même préfixe, sinon toutes les clés
    prefix = get_prefix(s)
    candidates = keys_by_prefix.get(prefix, keys)

    best_dist = None
    best_keys = []

    for k in candidates:
        d = levenshtein(s, k)
        if best_dist is None or d < best_dist:
            best_dist = d
            best_keys = [k]
        elif d == best_dist:
            best_keys.append(k)

    # si rien ou distance trop grande -> catégorie 5
    if best_dist is None or best_dist > max_dist:
        return 5

    # une seule meilleure clé
    if len(best_keys) == 1:
        return stratum_conv_dict[best_keys[0]]

    # plusieurs meilleures clés : prendre la plus fréquente dans df["STRATUM"]
    def freq(k):
        return stratum_freq.get(k, 0)

    best_key = max(best_keys, key=freq)
    return stratum_conv_dict[best_key]

# ------------------------------------------
# 4. application sur les valeurs uniques
# ------------------------------------------
unique_codes = df["STRATUM"].unique()
resolved_map = {code: closest_key(code) for code in unique_codes}

df["STRATUM"] = df["STRATUM"].map(resolved_map)

to_oh.append("STRATUM")

In [23]:
#Encoding
from typing import List, Dict

def target_encode(
    df: pd.DataFrame,
    target_col: str,
    cat_cols: List[str],
) -> pd.DataFrame:
    """
    Target encoding simple :
    - fit sur les lignes où target_col n'est pas NaN (train)
    - applique l'encodage à tout le df (train + test)
    - les catégories non vues prennent la moyenne globale du train
    """
    df = df.copy()
    
    # mask train = là où la cible est observée
    mask_train = df[target_col].notna()
    
    # moyenne globale du train
    global_mean = df.loc[mask_train, target_col].mean()
    
    for col in cat_cols:
        # moyenne de la target par catégorie (calculée uniquement sur le train)
        means = (
            df.loc[mask_train]
              .groupby(col, observed=True)[target_col]
              .mean()
        )
        
        df[col] = df[col].map(means).fillna(global_mean)
    
    return df


In [25]:
for c in to_oh: #One Hot pour AG/catboost
    df[c]=df[c].apply(lambda x : f"cat_{str(x)}")

In [27]:
df = robust_scale(df,"MathScore",to_rs)

In [29]:
final_cols+=to_rs
final_cols+=to_oh
final_cols.append("MathScore")

In [ ]:
df_final.to_csv("../../data/df_processed.csv")